# Pokémon Battle Prediction: final submissions
**Team Random Forretress**

This notebook clones our GitHub repository and runs the training script of each of our three selected submissions, the second, third and fourth versions. Each run writes its submission file to the output folder. All the feature and model code lives in the repository; the notebook only runs it.

## Setup
We clone the repository, point our data loader at the competition files and check the library versions.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/FrascaHoryzont/Pokemon-Battle-Prediction.git"
BRANCH = "clearcode"
REPO = Path("Pokemon-Battle-Prediction").resolve()

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--format=%h %ad %s", "--date=short"],
                     capture_output=True, text=True).stdout)

In [ ]:
# On Kaggle the competition files are under /kaggle/input; elsewhere DATA_DIR must point to them.
if "DATA_DIR" not in os.environ:
    found = sorted(Path("/kaggle/input").glob("**/train.jsonl"))
    if not found:
        raise FileNotFoundError("train.jsonl not found: attach the competition data or set DATA_DIR")
    os.environ["DATA_DIR"] = str(found[0].parent)
print("Data folder:", os.environ["DATA_DIR"])

OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
print("Output folder:", OUT)

import catboost, lightgbm, sklearn, xgboost
print("scikit-learn", sklearn.__version__, "| LightGBM", lightgbm.__version__,
      "| XGBoost", xgboost.__version__, "| CatBoost", catboost.__version__)

In [ ]:
def run_version(name):
    """Run one version's train.py from the repository and copy its submission to the output folder."""
    committed = (REPO / f"{name}_version" / "submission.csv").read_text()
    proc = subprocess.Popen([sys.executable, "-u", f"{name}_version/train.py"], cwd=REPO,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"{name}_version/train.py failed")
    target = OUT / f"submission_{name}_version.csv"
    shutil.copy(REPO / f"{name}_version" / "submission.csv", target)
    (OUT / f"committed_{name}_version.csv").write_text(committed)
    print("Saved", target)

## Submission 1: second version
LightGBM (small trees, 8 leaves) and CatBoost (depth 6), averaged. 470 features: the turn 30 state with P2's unrecorded knock outs counted, statuses, reserves and team strength; Spikes still up on each side; which species are alive and which moves each side has used; survivor matchups, speeds, HP per species and the duel between the active Pokémon; P2's hidden Pokémon estimated by looking its team up among the training teams; and a knock out detector trained on P1's recorded faints. Cross validated accuracy: 68.64%.

In [ ]:
run_version("second")

## Submission 2: third version
XGBoost, LightGBM and CatBoost blended with fixed weights (60/30/10) and a decision threshold of 0.505, bagged over 15 folds. The second version's features plus Gen 2 mechanics (Snorlax, Rest against hostile sleep, Spikes immunities and spin blockers, the physical and special split, missing resistances, volatile effects, HP momentum and move archetypes): 565 candidates, of which the top 200 by XGBoost gain are kept. Cross validated accuracy with the features selected inside each fold: 68.54%.

A rerun can differ from the submitted file on a few percent of borderline battles with the same accuracy: one feature sums probabilities over a Python set, whose order changes between runs, and XGBoost is sensitive to the resulting last digit differences.

In [ ]:
run_version("third")

## Submission 3: fourth version
The same features and base models as the third version, combined by a stacking meta learner instead of fixed weights: an L2 regularised logistic regression trained on the base models' out of fold probabilities, whose intercept also sets the decision threshold. It learns near equal weights and a threshold of 0.50. Cross validated accuracy with the features selected inside each fold: 68.55%. The same note on reruns applies.

In [ ]:
run_version("fourth")

## Check
Each file must have 5,000 rows, the columns `battle_id` and `player_won`, and the same ids as the sample submission. We also show how often each rerun agrees with the file committed in the repository.

In [ ]:
import pandas as pd

sample = pd.read_csv(REPO / "sample_submission.csv")
for name in ("second", "third", "fourth"):
    sub = pd.read_csv(OUT / f"submission_{name}_version.csv")
    committed = pd.read_csv(OUT / f"committed_{name}_version.csv")
    assert list(sub.columns) == ["battle_id", "player_won"] and len(sub) == 5000
    assert sub["battle_id"].tolist() == sample["battle_id"].tolist()
    assert set(sub["player_won"].unique()) <= {0, 1}
    agree = (sub["player_won"].to_numpy() == committed["player_won"].to_numpy()).mean()
    print(f"{name} version: valid, P1 predicted to win {sub['player_won'].mean():.1%}, "
          f"agrees with the committed file on {agree:.1%} of battles")